# Example: delay CLEAN on ASKAP data

`jolly_tractor clean` on the same data as the [notch example](nulling_example.ipynb), so the
two can be compared baseline by baseline. The figures on this page are made from the data
each time the documentation is built.

In [ ]:
import logging
import shutil
import tempfile
from pathlib import Path

from casacore.tables import table
from IPython.display import Image, display

from jolly_roger.logging import logger

logger.setLevel(logging.WARNING)

# The test data live in tests/data at the top of the repository
DATA = next(
    parent / "tests" / "data"
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "tests" / "data").is_dir()
)
WORK = Path(tempfile.mkdtemp())


def unpack(name: str) -> Path:
    """Unpack a measurement set from tests/data into a scratch directory"""
    shutil.unpack_archive(DATA / f"{name}.zip", WORK / name)
    return WORK / name / name


def last_history(ms: Path) -> str:
    with (
        table(str(ms), ack=False) as tab,
        table(tab.getkeyword("HISTORY"), ack=False) as history,
    ):
        return history.getcol("MESSAGE")[-1]

## The data

Thirty minutes of an ASKAP EMU observation (SB47138, EMU_1141-55, beam 6), centred on
sunrise, on the baselines from ak01 to ak04, ak06, ak07 and ak32. After sunrise the Sun is
a bright object outside the field. On ak01–ak04 its delay is close to the field's; on the
other baselines it is 0.5–9.5 µs away.

## Cleaning the Sun

The equivalent command is
`jolly_tractor clean sb47138_sunrise.ms --target-objects sun --make-plots`.
In each figure the left column is before and the right column after. The top row is the
dynamic spectrum and the second row is delay against time.

In [ ]:
from jolly_roger.deconvolve import CleanTractorOptions, clean_tractor

ms = unpack("sb47138_sunrise.ms")
results = clean_tractor(
    ms_path=ms,
    clean_tractor_options=CleanTractorOptions(
        output_column="CLEANED_DATA", make_plots=True, number_of_plots=4
    ),
)
print(last_history(ms))
for plot in results.output_plots:
    display(Image(filename=plot))

On ak01–ak04 the Sun's subtracted emission reaches the field's delay range, so CLEAN
flags those rows.

## Recovering flagged rows with fringe-rate CLEAN

`--rate-filter` CLEANs each run of flagged rows in fringe rate, channel by channel, and
subtracts only the components in the Sun's fringe-rate band. As in the
[notch example](nulling_example.ipynb), this needs a longer run on the short ak01–ak04
baseline: here 2 hours from just before sunrise.

The equivalent command is
`jolly_tractor clean sb47138_ak01_ak04.ms --target-objects sun --rate-filter --make-plots`.
The comparison figure has a third row, in fringe rate and delay, and each filtered segment
is plotted before and after.

In [ ]:
ms = unpack("sb47138_ak01_ak04.ms")
results = clean_tractor(
    ms_path=ms,
    clean_tractor_options=CleanTractorOptions(
        output_column="CLEANED_DATA",
        rate_filter=True,
        make_plots=True,
        number_of_plots=1,
    ),
)
print(last_history(ms))
for plot in [
    *results.output_plots,
    *sorted((ms.parent / "plots").glob("rate_filter_*.png")),
]:
    display(Image(filename=plot))